# ETL y Limpieza de Datos Logísticos



In [1]:
# Instalar pandas solo si estás en un entorno tipo JupyterLite / navegador sin librería preinstalada
# %mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 2.292 seconds
  Name    Version  Build                Channel
-------------------------------------------------------------
+ pandas  3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
- pip     26.2.2   pyh145f28c_0         conda-forge


In [2]:
import pandas as pd
import numpy as np
# Librerías principales para manipulación de datos y lógica numérica


### 1. Carga de Datos y Diagnóstico Inicial

En esta etapa realizamos una exploración inicial del dataset para evaluar la estructura de las columnas, identificar tipos de datos inconsistentes, detectar valores nulos (`NaN`) y buscar posibles anomalías o registros duplicados antes de iniciar la transformación.

In [3]:
# Cargar el dataset crudo (raw) antes de aplicar transformaciones
df = pd.read_csv("pedidos_logistica_raw.csv")
# Mostrar las primeras 50 filas para inspeccionar formatos y anomalías iniciale
df.head(50)

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega
0,ORD-10000,2026-07-03,NaN,Camila Martínez,Metropolitana,Vestuario,-24280.0,2990.0,NaN
1,ORD-10001,09/07/2026,2026-07-24,valentina fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Metropolitana,Belleza,119180.0,6990.0,Entregado
3,ORD-10003,02/09/2026,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,$370.400,6990.0,Entregado
4,ORD-10004,08/09/2026,NaN,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En tránsito
5,ORD-10005,2026-08-21,2026-08-30,francisca díaz,Biobío,Belleza,153130.0,4990.0,Entregado
6,ORD-10006,2026-09-05,2026-09-12,Joaquín Martínez,Araucanía,Hogar,157240.0,3990.0,Entregado
7,ORD-10007,2026-08-04,NaN,Constanza Rojas,Biobío,belleza,96360.0,6990.0,En tránsito
8,ORD-10008,2026-06-10,2026-06-12,JOAQUÍN CONTRERAS,Metropolitana,Belleza,195780.0,0.0,Entregado
9,ORD-10009,2026-06-22,2026-06-25,Benjamín Rojas,RM,Vestuario,416580.0,0.0,Entregado


In [4]:
df.info()
# Revisar estructura general, tipos de datos y valores no nulos por columna
print("Duplicados encontrados:", df.duplicated().sum())
# Verificar si existen filas completamente duplicadas

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pedido_id       500 non-null    str    
 1   fecha_pedido    500 non-null    str    
 2   fecha_entrega   351 non-null    str    
 3   cliente         486 non-null    str    
 4   region          500 non-null    str    
 5   categoria       489 non-null    str    
 6   monto_venta     479 non-null    str    
 7   costo_envio     467 non-null    float64
 8   estado_entrega  487 non-null    str    
dtypes: float64(1), str(8)
memory usage: 19.6 KB
Duplicados encontrados: 0


### 2. Transformación y Limpieza 



In [5]:
# Limpiar formato de texto en monto_venta (espacios, signo $, sufijo .0 y puntos de miles)
df["monto_venta"] = (
    df["monto_venta"]
    .str.strip()
    .str.replace("$", "", regex=False)
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(".", "", regex=False)
)

#  Convertir a valor numérico; si hay algún valor no válido, queda como NaN
df["monto_venta"] = pd.to_numeric(df["monto_venta"], errors="coerce")
# Verificar los primeros 10 pedidos con su monto ya numérico
print(df[["pedido_id", "monto_venta"]].head(10))


   pedido_id  monto_venta
0  ORD-10000     -24280.0
1  ORD-10001     371470.0
2  ORD-10002     119180.0
3  ORD-10003     370400.0
4  ORD-10004     192680.0
5  ORD-10005     153130.0
6  ORD-10006     157240.0
7  ORD-10007      96360.0
8  ORD-10008     195780.0
9  ORD-10009     416580.0


In [6]:
# Estandarizar texto en categoria (quitar espacios y formato Título) conservando nulos reales
df["categoria"] = df["categoria"].astype(str).str.strip().str.title()
df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega
0,ORD-10000,2026-07-03,NaN,Camila Martínez,Metropolitana,Vestuario,-24280.0,2990.0,NaN
1,ORD-10001,09/07/2026,2026-07-24,valentina fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Metropolitana,Belleza,119180.0,6990.0,Entregado
3,ORD-10003,02/09/2026,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado
4,ORD-10004,08/09/2026,NaN,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En tránsito


### 3. Normalización de Regiones y Clientes



In [7]:
# Normalizar regiones para evitar duplicados por abreviaturas, tildes o mayúsculas/minúsculas
df["region"] = df["region"].astype(str).str.strip().str.lower()

diccionario = {
    "rm": "Región Metropolitana",
    "metropolitana": "Región Metropolitana",
    "region metropolitana": "Región Metropolitana",
}

# Formatear nombres de clientes conservando valores nulos reales
df["region"] = df["region"].replace(diccionario)
df["region"] = df["region"].str.title()
df["region"] = df["region"].replace({"Valparaiso": "Valparaíso"})

df["cliente"] = df["cliente"].astype(str).str.strip().str.title()

print(df["region"].unique())
df[["cliente", "region"]].head()

<StringArray>
['Región Metropolitana',           'Valparaíso',            'Araucanía',
               'Biobío',            'Los Lagos',                'Maule',
             'Coquimbo',            'O'Higgins']
Length: 8, dtype: str


,cliente,region
0,Camila Martínez,Región Metropolitana
1,Valentina Fuentes,Valparaíso
2,Diego González,Región Metropolitana
3,Tomás Silva,Región Metropolitana
4,Nicolás Fuentes,Araucanía


### 4. Creación de KPIs y Manejo de Anomalías



In [8]:
# Calcular ingreso_total solo para ventas positivas; devoluciones/ajustes o nulos quedan en 0
df["ingreso_total"] = np.where(
    df["monto_venta"] > 0,
    df["monto_venta"] + df["costo_envio"],
    0,
)

# Clasificar el tipo de movimiento según si el monto de venta es positivo
df["tipo_movimiento"] = np.where(
    df["monto_venta"] > 0, "Venta", "Devolución / Ajuste"
)
df[["monto_venta", "ingreso_total", "tipo_movimiento"]].head()

,monto_venta,ingreso_total,tipo_movimiento
0,-24280.0,0.0,Devolución / Ajuste
1,371470.0,378460.0,Venta
2,119180.0,126170.0,Venta
3,370400.0,377390.0,Venta
4,192680.0,199670.0,Venta


### 5. Procesamiento de Fechas y Cálculo de Lead Time (Días de Entrega)



In [9]:
def parsear_fecha_segura(serie):
    ###Convierte fechas en formato YYYY-MM-DD o DD/MM/YYYY de forma segura a datetime.
  s = serie.astype(str).str.strip()
  mask_iso = s.str.match(r"^\d{4}-\d{2}-\d{2}$")
  mask_dmy = s.str.match(r"^\d{2}/\d{2}/\d{4}$")

  fechas = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
  fechas[mask_iso] = pd.to_datetime(
      s[mask_iso], format="%Y-%m-%d", errors="coerce"
  )
  fechas[mask_dmy] = pd.to_datetime(
      s[mask_dmy], format="%d/%m/%Y", errors="coerce"
  )
  return fechas

# Parsear fechas de pedido y entreg
df["fecha_pedido"] = parsear_fecha_segura(df["fecha_pedido"])
df["fecha_entrega"] = parsear_fecha_segura(df["fecha_entrega"])

# Calcular diferencia en días entre fecha de entrega y fecha de pedido
df["dias_entrega"] = (df["fecha_entrega"] - df["fecha_pedido"]).dt.days

# Crear versiones formateadas en texto (DD/MM/YYYY) para visualización/reporte
df["fecha_pedido_str"] = df["fecha_pedido"].dt.strftime("%d/%m/%Y")
df["fecha_entrega_str"] = df["fecha_entrega"].dt.strftime("%d/%m/%Y")

df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str
0,ORD-10000,2026-07-03,NaT,Camila Martínez,Región Metropolitana,Vestuario,-24280.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,03/07/2026,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto,378460.0,Venta,15.0,09/07/2026,24/07/2026
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,119180.0,6990.0,Entregado,126170.0,Venta,6.0,13/08/2026,19/08/2026
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026
4,ORD-10004,2026-09-08,NaT,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En tránsito,199670.0,Venta,NaN,08/09/2026,NaN


limpiar los dias negativos 

In [10]:
# Conservar solo días de entrega >= 0; los valores negativos se dejan como NaN
df["dias_entrega_limpio"] = df["dias_entrega"].where(df["dias_entrega"] >= 0)

df.head()

,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str,dias_entrega_limpio
0,ORD-10000,2026-07-03,NaT,Camila Martínez,Región Metropolitana,Vestuario,-24280.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,03/07/2026,NaN,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto,378460.0,Venta,15.0,09/07/2026,24/07/2026,15.0
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,119180.0,6990.0,Entregado,126170.0,Venta,6.0,13/08/2026,19/08/2026,6.0
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026,12.0
4,ORD-10004,2026-09-08,NaT,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En tránsito,199670.0,Venta,NaN,08/09/2026,NaN,NaN


Limpieza de estado_entrega con el mismo formato

In [11]:
# Estandarizar estado_entrega conservando los valores nulos reales
mask_estado = df["estado_entrega"].notna()
df.loc[mask_estado, "estado_entrega"] = (
    df.loc[mask_estado, "estado_entrega"].astype(str).str.strip().str.title()
)

print("Estados únicos:", df["estado_entrega"].unique())
df.head()

Estados únicos: <StringArray>
[nan, 'Devuelto', 'Entregado', 'En Tránsito', 'Cancelado', 'Pendiente']
Length: 6, dtype: str


,pedido_id,fecha_pedido,fecha_entrega,cliente,region,categoria,monto_venta,costo_envio,estado_entrega,ingreso_total,tipo_movimiento,dias_entrega,fecha_pedido_str,fecha_entrega_str,dias_entrega_limpio
0,ORD-10000,2026-07-03,NaT,Camila Martínez,Región Metropolitana,Vestuario,-24280.0,2990.0,NaN,0.0,Devolución / Ajuste,NaN,03/07/2026,NaN,NaN
1,ORD-10001,2026-07-09,2026-07-24,Valentina Fuentes,Valparaíso,Calzado,371470.0,6990.0,Devuelto,378460.0,Venta,15.0,09/07/2026,24/07/2026,15.0
2,ORD-10002,2026-08-13,2026-08-19,Diego González,Región Metropolitana,Belleza,119180.0,6990.0,Entregado,126170.0,Venta,6.0,13/08/2026,19/08/2026,6.0
3,ORD-10003,2026-09-02,2026-09-14,Tomás Silva,Región Metropolitana,Hogar,370400.0,6990.0,Entregado,377390.0,Venta,12.0,02/09/2026,14/09/2026,12.0
4,ORD-10004,2026-09-08,NaT,Nicolás Fuentes,Araucanía,Hogar,192680.0,6990.0,En Tránsito,199670.0,Venta,NaN,08/09/2026,NaN,NaN


In [12]:
# Convertir columnas numéricas a tipo entero nullable (Int64) para admitir nulos sin decimales .0
cols_enteras = [
    "monto_venta",
    "costo_envio",
    "ingreso_total",
    "dias_entrega_limpio",
]
for col in cols_enteras:
  df[col] = df[col].round().astype("Int64")

# Exportar el dataset limpio una sola vez con codificación UTF-8 con BOM para compatibilidad
df.to_csv("pedidos_logistica_limpios.csv", index=False, encoding="utf-8-sig")
print("¡Archivo 'pedidos_logistica_limpios.csv' exportado con éxito!")

¡Archivo 'pedidos_logistica_limpios.csv' exportado con éxito!


### 6. Exportación del Dataset Limpio

*(Escribe aquí tus conclusiones finales antes de exportar el archivo CSV)*

In [13]:
df.to_csv("pedidos_logistica_limpios.csv", index=False)
print("¡Archivo exportado con éxito!")

¡Archivo exportado con éxito!


Conclusión del proceso ETL:
Se procesaron los 500 registros del dataset logístico original y se exportó el archivo limpio con 15 columnas, manteniendo la integridad de los montos y estandarizando formatos antes del análisis.

Limpieza financiera: Se normalizó monto_venta (conservando valores flotantes con .0, convirtiendo formatos con $ y puntos de miles y manteniendo los 21 nulos reales) y se crearon tipo_movimiento (472 Venta y 28 Devolución / Ajuste) e ingreso_total ($102.537.860 en total).

Normalización de texto: Se unificaron region (8 regiones), categoria (6 categorías sin duplicados por mayúsculas/minúsculas) y estado_entrega (Entregado, En Tránsito, Pendiente, Cancelado, Devuelto y nulos reales).

Tiempos de entrega: Se parsearon fechas y se creó dias_entrega_limpio (244 registros válidos >= 0, con mediana de 11.0 días y promedio de 54.2 días), aislando 107 casos negativos y 149 pedidos sin fecha de entrega.